In [1]:
# !pip3 install torch numpy joblib scikit-learn

import torch
import torch.nn as nn
import numpy as np

import joblib
from sklearn.decomposition import PCA

from robot_state import RobotData, ObjectState
from RobotPolicyNet import RobotPolicy
import torch.optim as optim

/home/hoang-dung/upwork_project/robot_hand_control/orca_hand_ws/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
"""
Load trained models
"""
pca2 = joblib.load("/home/hoang-dung/upwork_project/robot_hand_control/orca_hand_ws/scripts/train/hand_synergy_pca.pkl")
policy = RobotPolicy()
policy.load_state_dict(torch.load("robot_policy_model.pth"))

/home/hoang-dung/upwork_project/robot_hand_control/orca_hand_ws/.venv/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.4.1.post1 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


<All keys matched successfully>

In [3]:
def cvt_state_to_input(state):
    """
    Convert the state to a format suitable for the model input.
    """
    robot_state = RobotData.from_dict(state)
    return robot_state.to_model_input()

def cvt_state_to_output(state):
    """
    Convert the model output to a format suitable for the environment.
    """
    robot_state = RobotData.from_dict(state)
    return robot_state.to_model_output()


In [4]:
"""
Load json data
"""
import json
data_path = "/home/hoang-dung/upwork_project/robot_hand_control/orca_hand_ws/robot_data_test.json"
def load_json(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

data = load_json(data_path)
testing_pairs = []
# Create input - output pairs
for i in range(len(data)-1):
    if data[i]['action']  == 'none' or data[i]['action'] == 'None':
        continue

    if data[i]['action'] == data[i+1]['action']:
        if not data[i]['objects'] and not data[i+1]['objects']:
            continue
        # In a same action
        if not data[i+1]['objects'] and data[i]['objects']:
            # duplicate previous objects if empty
            data[i+1]['objects'] = data[i]['objects']

        if not data[i]['objects'] and data[i+1]['objects']:
            # duplicate next objects if empty
            data[i]['objects'] = data[i+1]['objects']

        if data[i]['objects']:
            testing_pairs.append((
                cvt_state_to_input(data[i]),
                cvt_state_to_output(data[i+1])
            ))

    elif data[i]['action'] != data[i+1]['action']:
        testing_pairs.append((
            cvt_state_to_input(data[i]),
            cvt_state_to_output(data[i])
        ))


data: {'action': 'grab', 'objects': [{'name': 'cup', 'position': [[0.37031930490708076, 0.3162881526933226, 0.1890619684418795], [0.3739054901936147, 0.3133614286767092, 0.16422312282708562], [0.3770516408978147, 0.3161180687801391, 0.13645718478636804], [0.38519663471269305, 0.3709348887790373, 0.17173352134105097], [0.3927158086195203, 0.31242202280641723, 0.16847451160547677], [0.39588961600500555, 0.3183304531116155, 0.1396878143809292], [0.40509997162037364, 0.3632007936565987, 0.17835710411284683], [0.41045136226320666, 0.3262063947080455, 0.16658344403400016], [0.442050737468907, -0.4482952228579697, 0.48455457218389697]]}], 'eef_position': [0.2814064916322352, 0.25660774310559076, 0.3208800016711012, -1.76921532092702, -1.4579747538289292, -0.552526253206716], 'joint_positions': [-1.5322068378259874, 3.4271149108320462, 0.02868155937393219, 1.6065102346168783, 1.0353847178465434, -1.4733384265998015], 'finger_positions': [26.305242825336393, 61.94882697152528, -3.26002536896953

In [5]:
for data in testing_pairs:
    res = policy.forward([data[0]])

    print("Predicted:", res)
    print("Ground truth:", data[1])

Predicted: tensor([[-0.1166, -0.0504,  0.1561, -2.1006, -1.4545, -0.2520,  0.5871,  0.7045,
         -0.0522, -0.1032,  0.9924, -0.4364,  0.7276, -0.0249,  0.6726,  0.8950,
          1.0163, -0.0660,  1.9615,  0.9559, -0.2367, -0.4856]],
       grad_fn=<AddmmBackward0>)
Ground truth: tensor([-0.1186, -0.0580,  0.1470, -2.0902, -1.4016, -0.2346,  0.6141,  0.6827,
        -0.0398, -0.1083,  0.9972, -0.4566,  0.7416, -0.0067,  0.6675,  0.9109,
         1.0369, -0.0541,  1.9411,  0.9488, -0.2271, -0.4787])
Predicted: tensor([[-0.1010, -0.0490,  0.1647, -2.0459, -1.4333, -0.2270,  0.5932,  0.7071,
         -0.0508, -0.1073,  1.0103, -0.4474,  0.7342, -0.0139,  0.6714,  0.8959,
          1.0280, -0.0563,  1.9577,  0.9635, -0.2278, -0.4779]],
       grad_fn=<AddmmBackward0>)
Ground truth: tensor([-0.1186, -0.0580,  0.1470, -2.0902, -1.4016, -0.2346,  0.6141,  0.6827,
        -0.0398, -0.1083,  0.9972, -0.4566,  0.7416, -0.0067,  0.6675,  0.9109,
         1.0369, -0.0541,  1.9411,  0.9488, -0.